# 05 · Persistent and CLC scheduling

So far each CTA computed one output tile and exited. **Persistent** scheduling keeps CTAs resident and feeds them many tiles, amortising prologue and epilogue across all of them.

Then the question becomes *how* tiles are assigned — a fixed order, or on demand at runtime. The dynamic option is strictly more flexible. This notebook measures what that flexibility costs when you do not need it.

## 1. The question

Two schedulers, same work:

- **gemm 3** — `StaticPersistentTileScheduler`. Tiles assigned in a fixed order, zero scheduler overhead, vulnerable to load imbalance.
- **gemm 3_1** — dynamic CLC (cooperative launch control). CTAs request tiles at runtime, so a fast CTA can pick up work from a slow one.

**The question:** what does work-stealing cost when the load is already balanced?

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        610.57.04
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

Persistent kernels need **block-level containers**: `tma_block`, `mma_block`, `epilogue_block` wrap each role's entire multi-tile pass, so you can see per-CTA residency rather than just per-tile work.

The dynamic variant adds ranges that exist only because CLC exists — `tma_clc_wait`, `mma_clc_wait`, `clc_prod_acquire`. **The overhead of a mechanism is directly nameable**, which is what makes its cost measurable rather than inferred from a wall-clock difference.

In [2]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fp16_gemm_3_1_iket_detailed_marks.py"),
    limit=14,
)

fp16_gemm_3_1_iket_detailed_marks.py: 67 IKET call sites (showing first 14)
------------------------------------------------------------------------
>  119      cute.experimental.iket.range_push("prologue")
       ...
>  370      cute.experimental.iket.range_pop()
       ...
>  380          cute.experimental.iket.range_push("tma_block")
       ...
>  385              cute.experimental.iket.range_push("tma_tile")
       ...
>  403                  cute.experimental.iket.range_push("tma_iter")
       ...
>  405                  cute.experimental.iket.range_push("prod_acquire")
       ...
>  407                  cute.experimental.iket.range_pop()
       ...
>  410                  cute.experimental.iket.mark("tma_issue")  # async op -> mark
       ...
>  425                  cute.experimental.iket.range_pop()
       ...
>  429                  cute.experimental.iket.range_push("tma_clc_wait")
       ...
>  434                  cute.experimental.iket.range_pop()
       ...
>  436          

## 3. Run it

In [3]:
for tag, src in [("gemm3", "fp16_gemm_3_iket_detailed_marks.py"),
                 ("gemm3_1", "fp16_gemm_3_1_iket_detailed_marks.py")]:
    cmd = (
        f"run-iket --output-dir out/nb05_{tag} --clobber profile --postprocess all -- "
        f"python {nbhelp.kernel('instrumented', src)} --mnk 4096,4096,512"
    )
    print(f"===== {tag} =====")
    check_env.run_or_show(cmd, env, est="about 1 minute each")

===== gemm3 =====
$ run-iket --output-dir out/nb05_gemm3 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_3_iket_detailed_marks.py --mnk 4096,4096,512

[run-iket] ============= Dry run application first to collect some info... =============


Running Blackwell fp16 GEMM example 3 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_3_iket_detailed_marks.py:606: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/static_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.PersistentTileSchedulerParams(
PASS
[run-iket] Auto-computed context buffer size: 0.00 GB (1,323,475 bytes)
[run-iket] ============= Run application for real profiling... =============


Running Blackwell fp16 GEMM example 3 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_3_iket_detailed_marks.py:606: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/static_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.PersistentTileSchedulerParams(
PASS


[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3/iket_pid_0x391f07.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3/iket_pid_0x391f07.trace.json


[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3/iket_pid_0x391f07.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3/iket_pid_0x391f07.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3 && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x391f07.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3/gen-config
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3/iket
===== gemm3_1 =====
$ run-iket --output-dir out/nb05_gemm3_1 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_3_1_i

[run-iket] ============= Dry run application first to collect some info... =============


Running Blackwell fp16 GEMM example 3_1 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_3_1_iket_detailed_marks.py:730: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
PASS
[run-iket] Auto-computed context buffer size: 0.02 GB (20,904,755 bytes)
[run-iket] ============= Run application for real profiling... =============


Running Blackwell fp16 GEMM example 3_1 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_3_1_iket_detailed_marks.py:730: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
PASS


[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1/iket_pid_0x391f88.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1/iket_pid_0x391f88.trace.json


[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1/iket_pid_0x391f88.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1/iket_pid_0x391f88.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1 && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x391f88.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1/gen-config
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb05_gemm3_1/iket


## 4. The trace

Each CTA stays resident and runs one role across its whole tile set:

In [4]:
nbhelp.figure("nb05/gemm3_static_persistent_trace.png", "Static persistent: per-CTA role spans across the full tile set.")

The dynamic scheduler adds bands that simply do not exist in the static kernel — this is the price of flexibility, made visible:

In [5]:
nbhelp.figure("nb05/gemm3_1_clc_scheduler_overhead.png", "New CLC bands: the request/wait handshake.")

In [6]:
try:
    trace = ts.load(ts.find_trace("out/nb05_gemm3_1"))
    ts.print_summary(ts.summarize(trace), top=12)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_kernel_TiledMMA_ThrLayoutVMNK21111000_PermutationMNK____MMAAtom_ThrID21_ShapeMNK25625616_TVLayoutA2128161281256_TVLayoutB2128161281256_TVLayoutC21282561281256_cutlasscutenv_0   grid 32x16x1  block 224x1x1  warps traced 1036
warp-lifetime span: 16.6 us

range                      count      median         p90       total
--------------------------------------------------------------------
epilogue_block               592     12.448u     15.136u    8033.12u
epi_tile                    2048      3.136u      5.568u    7550.46u
epi_iter                   16384      0.256u      0.384u    4551.23u
acc_wait                    2048      0.256u      2.752u    1881.25u
mma_block                    148     12.352u     14.976u    1831.94u
tma_block                    148     10.304u     13.216u    1708.48u
tma_tile                     512      3.168u      4.608u    1660.96u
sched_block                   74     12.576u     15.232u    1011.39u
mma_clc_wait                 512      0.78

## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,512`:

| | gemm 3 (static) | gemm 3_1 (dynamic CLC) |
|---|---|---|
| Warp-lifetime span | **14.37 µs** | 16.54 µs (+15%) |
| Warps traced | 888 | 1,036 |
| `tma_block` median | 9.024 µs | 10.208 µs |
| `mma_block` median | 5.408 µs | 12.304 µs |
| `epilogue_block` median | 10.592 µs | 12.416 µs |


**The CLC-only ranges, which have no static counterpart:**

| Range | Median | Count |
|---|---|---|
| `clc_prod_acquire` | 2.832 µs | 256 |
| `tma_clc_wait` | 1.376 µs | 512 |
| `mma_clc_wait` | 0.752 µs | 512 |
| `sched_clc_wait` | 0.160 µs | 256 |


Dynamic scheduling costs 15% here, and the trace does not merely say *that* — it says **where**: a 2.832 µs scheduler-to-pipeline handshake plus a per-tile-transition wait on both the TMA and MMA sides.

**This is not a verdict against CLC.** The workload is balanced, so work-stealing has nothing to steal and only the overhead remains. On an uneven distribution — a shape where tiles-per-CTA does not divide evenly, or a grouped GEMM — the same mechanism pays for itself. What the profile gives you is the *price*, so you can judge whether your workload will buy anything with it.

Note also that every kernel from 02 onward runs a similar per-tile pipeline. The gemm 2 → 3 → 3_1 progression is almost entirely a story about the rows *around* the pipeline: launch, prologue, epilogue, scheduler.

In [7]:
nbhelp.figure("nb05/gemm2_3_3_1_per_tile_table.png", "Per-tile critical path across the three variants.")

**Next:** [06 · Preferred and fallback clusters](06_cluster_shapes.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*